# Paper Theater Photoshop Pipeline

This notebook adds a **Photoshop-ready export stage** to your paper theater pipeline.

It is designed around the problem you called out:
- every object should become a **full-canvas layer**
- masks should also be **full-canvas**
- Photoshop should receive a **clean package** with transparent cutouts, masks, prompts, and a layer manifest
- you can optionally generate a **Photoshop JSX importer** that builds a layered PSD automatically

## What this notebook does

1. Loads an image and your segmentation results  
2. Converts every object mask into a **full-size mask**  
3. Exports **full-size transparent RGBA layers**  
4. Builds a **manifest.json** with prompts and layer order  
5. Writes a **Photoshop JSX script** to import everything into one PSD  
6. Packages everything into a folder that can be used with:
   - manual Photoshop editing
   - Generative Fill
   - ChatGPT + Photoshop workflows
   - future round-trip automation

## Expected input

This notebook supports either:
- a Python variable named `merged_segmented` already in memory, or
- a JSON file containing a list of objects like:

```json
[
  {
    "label": "temple",
    "bbox": [x1, y1, x2, y2],
    "layer": 2,
    "mask_path": "path/to/mask.png"
  }
]
```

## Notes

- The notebook does **not** require Photoshop to export the assets.
- The optional JSX script runs inside desktop Photoshop.
- If your current pipeline already has `layer_assignments`, `cleaned_mask`, or `openai_results`, you can plug them in directly.

In [ ]:

# If you are in Colab, uncomment:
# !pip install pillow numpy matplotlib opencv-python

import json
import csv
import shutil
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
from PIL import Image, ImageOps

try:
    import cv2
except Exception:
    cv2 = None

In [ ]:

# ----------------------------
# Configuration
# ----------------------------

PROJECT_NAME = "paper_theater_photoshop_pipeline"

# Main inputs
IMAGE_PATH = Path("input/image.png")              # change this
SEGMENTS_JSON = Path("input/segments.json")       # optional
MASKS_ARE_FULLSIZE = False                        # set True if mask files are already full-canvas
BACKGROUND_LABELS = {"sky", "cloud", "sun", "moon"}
STRUCTURED_LABELS = {"temple", "bridge", "building", "gate", "pagoda", "mountain"}
ORGANIC_LABELS = {"tree", "bush", "plant", "flower", "deer", "bird", "person"}

# Output folder
OUT_ROOT = Path("output") / PROJECT_NAME

# Optional cleanup
SMOOTH_KERNEL = 5
MIN_COMPONENT_AREA = 64

# Depth ordering
DEFAULT_LAYER_IF_MISSING = 999

# Export options
SAVE_PREVIEW_CONTACT_SHEET = True
SAVE_ZIP_PACKAGE = True
GENERATE_JSX_IMPORTER = True

In [ ]:

# ----------------------------
# Helper functions
# ----------------------------

def ensure_dir(p: Path) -> Path:
    p.mkdir(parents=True, exist_ok=True)
    return p

def load_image(path: Path) -> Image.Image:
    if not path.exists():
        raise FileNotFoundError(f"Image not found: {path}")
    return Image.open(path).convert("RGBA")

def load_mask(mask_path: Path) -> np.ndarray:
    m = Image.open(mask_path).convert("L")
    return np.array(m)

def binarize_mask(mask: np.ndarray, threshold: int = 127) -> np.ndarray:
    return (mask > threshold).astype(np.uint8) * 255

def smooth_mask(mask: np.ndarray, kernel: int = SMOOTH_KERNEL) -> np.ndarray:
    if cv2 is None or kernel <= 1:
        return mask
    k = np.ones((kernel, kernel), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k)
    return mask

def remove_small_components(mask: np.ndarray, min_area: int = MIN_COMPONENT_AREA) -> np.ndarray:
    if cv2 is None:
        return mask
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats((mask > 0).astype(np.uint8), 8)
    out = np.zeros_like(mask)
    for i in range(1, num_labels):
        area = stats[i, cv2.CC_STAT_AREA]
        if area >= min_area:
            out[labels == i] = 255
    return out

def cleanup_mask(mask: np.ndarray) -> np.ndarray:
    mask = binarize_mask(mask)
    mask = remove_small_components(mask)
    mask = smooth_mask(mask)
    return binarize_mask(mask)

def expand_crop_mask_to_full(mask_crop: np.ndarray, bbox: List[int], full_hw: Tuple[int, int]) -> np.ndarray:
    h, w = full_hw
    x1, y1, x2, y2 = map(int, bbox)
    full = np.zeros((h, w), dtype=np.uint8)
    crop_h, crop_w = mask_crop.shape[:2]
    target_h = max(0, y2 - y1)
    target_w = max(0, x2 - x1)

    if crop_h != target_h or crop_w != target_w:
        mask_crop_img = Image.fromarray(mask_crop)
        mask_crop = np.array(mask_crop_img.resize((target_w, target_h), Image.NEAREST))

    full[y1:y2, x1:x2] = mask_crop
    return full

def rgba_cutout_full_canvas(image_rgba: Image.Image, full_mask: np.ndarray) -> Image.Image:
    arr = np.array(image_rgba).copy()
    alpha = (full_mask > 0).astype(np.uint8) * 255
    arr[..., 3] = alpha
    return Image.fromarray(arr, mode="RGBA")

def mask_bbox(mask: np.ndarray) -> Optional[List[int]]:
    ys, xs = np.where(mask > 0)
    if len(xs) == 0 or len(ys) == 0:
        return None
    return [int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1]

def mask_area(mask: np.ndarray) -> int:
    return int((mask > 0).sum())

def sanitize_name(name: str) -> str:
    keep = []
    for ch in name.lower().strip():
        if ch.isalnum() or ch in ("_", "-"):
            keep.append(ch)
        elif ch in (" ", "/"):
            keep.append("_")
    out = "".join(keep).strip("_")
    return out or "layer"

def object_prompt(label: str, layer_idx: int) -> str:
    label_l = label.lower()
    if label_l in BACKGROUND_LABELS:
        return f"Complete the {label_l} as a full-canvas background layer, smooth and coherent, preserve original style, keep empty regions transparent only where the object does not exist."
    if label_l in STRUCTURED_LABELS:
        return f"Complete the hidden parts of the {label_l} naturally for a paper theater layer. Preserve architectural symmetry, clean silhouette, full-canvas layer, transparent background."
    if label_l in ORGANIC_LABELS:
        return f"Complete the full silhouette of the {label_l} for a layered paper theater scene. Keep natural contours, preserve texture, full-canvas layer, transparent background."
    return f"Complete the hidden parts of this {label_l} for a paper theater layer. Output should remain a full-canvas transparent layer with a clean silhouette."

def depth_group(layer_idx: int) -> str:
    if layer_idx <= 0:
        return "background"
    if layer_idx == 1:
        return "midground"
    return "foreground"

def save_np_mask(path: Path, mask: np.ndarray):
    Image.fromarray(mask).save(path)

def alpha_over_white(img_rgba: Image.Image) -> Image.Image:
    bg = Image.new("RGBA", img_rgba.size, (255, 255, 255, 255))
    return Image.alpha_composite(bg, img_rgba).convert("RGB")

In [ ]:

# ----------------------------
# Input loading
# ----------------------------

def load_segments_from_json(path: Path) -> List[Dict[str, Any]]:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise ValueError("segments.json must contain a list of objects")
    return data

def normalize_segments(raw_segments: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    norm = []
    for i, obj in enumerate(raw_segments):
        norm.append({
            "index": int(obj.get("index", i)),
            "label": str(obj.get("label", f"object_{i}")),
            "bbox": obj.get("bbox"),
            "layer": int(obj.get("layer", obj.get("depth_layer", DEFAULT_LAYER_IF_MISSING))),
            "mask_path": obj.get("mask_path"),
        })
    return norm

base_image = load_image(IMAGE_PATH)
W, H = base_image.size
print("Loaded image:", IMAGE_PATH)
print("Size:", W, "x", H)

if "merged_segmented" in globals():
    raw_segments = merged_segmented
    print("Using in-memory variable: merged_segmented")
elif SEGMENTS_JSON.exists():
    raw_segments = load_segments_from_json(SEGMENTS_JSON)
    print("Using JSON:", SEGMENTS_JSON)
else:
    raise FileNotFoundError(
        "No input segments found. Define `merged_segmented` in memory or create input/segments.json."
    )

segments = normalize_segments(raw_segments)
print("Loaded segments:", len(segments))
segments[:2]

In [ ]:

# ----------------------------
# Export Photoshop-ready assets
# ----------------------------

assets_dir = ensure_dir(OUT_ROOT / "assets")
masks_dir = ensure_dir(assets_dir / "masks_full")
layers_dir = ensure_dir(assets_dir / "layers_rgba")
preview_dir = ensure_dir(assets_dir / "previews")
manifest_dir = ensure_dir(OUT_ROOT / "manifest")
photoshop_dir = ensure_dir(OUT_ROOT / "photoshop")
reports_dir = ensure_dir(OUT_ROOT / "reports")

export_rows: List[Dict[str, Any]] = []

for i, obj in enumerate(segments):
    label = obj["label"]
    bbox = obj.get("bbox")
    layer_idx = int(obj.get("layer", DEFAULT_LAYER_IF_MISSING))
    mask_path = obj.get("mask_path")

    if mask_path is None:
        print(f"Skipping {i}: no mask_path")
        continue

    mask_path = Path(mask_path)
    if not mask_path.exists():
        print(f"Skipping {i}: missing mask file -> {mask_path}")
        continue

    raw_mask = load_mask(mask_path)
    raw_mask = binarize_mask(raw_mask)

    if MASKS_ARE_FULLSIZE:
        full_mask = raw_mask
    else:
        if bbox is None:
            raise ValueError(f"Object {i} has cropped mask but no bbox.")
        full_mask = expand_crop_mask_to_full(raw_mask, bbox, (H, W))

    full_mask = cleanup_mask(full_mask)

    bbox_full = mask_bbox(full_mask)
    area = mask_area(full_mask)
    if bbox_full is None or area == 0:
        print(f"Skipping {i}: empty mask after cleanup")
        continue

    rgba = rgba_cutout_full_canvas(base_image, full_mask)

    stem = f"{i:03d}_{sanitize_name(label)}_L{layer_idx}"
    mask_out = masks_dir / f"{stem}_mask.png"
    rgba_out = layers_dir / f"{stem}_layer.png"
    preview_out = preview_dir / f"{stem}_preview.jpg"

    save_np_mask(mask_out, full_mask)
    rgba.save(rgba_out)
    alpha_over_white(rgba).save(preview_out, quality=92)

    row = {
        "index": i,
        "name": stem,
        "label": label,
        "layer": layer_idx,
        "group": depth_group(layer_idx),
        "bbox": bbox_full,
        "area_px": area,
        "mask_path": str(mask_out.as_posix()),
        "rgba_path": str(rgba_out.as_posix()),
        "preview_path": str(preview_out.as_posix()),
        "prompt": object_prompt(label, layer_idx),
        "photoshop_notes": "Use full-canvas transparent layer. Keep object isolated. Fill missing occluded regions naturally.",
    }
    export_rows.append(row)

export_rows = sorted(export_rows, key=lambda x: (x["layer"], x["index"]))
print(f"Exported {len(export_rows)} Photoshop-ready assets to {assets_dir}")
export_rows[:3]

In [ ]:

# ----------------------------
# Save manifest files
# ----------------------------

manifest = {
    "project_name": PROJECT_NAME,
    "image_path": str(IMAGE_PATH.as_posix()),
    "canvas_width": W,
    "canvas_height": H,
    "num_layers": len(export_rows),
    "layers": export_rows,
}

manifest_json = manifest_dir / "manifest.json"
manifest_csv = reports_dir / "manifest.csv"

with open(manifest_json, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

with open(manifest_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            "index", "name", "label", "layer", "group", "bbox", "area_px",
            "mask_path", "rgba_path", "preview_path", "prompt", "photoshop_notes"
        ],
    )
    writer.writeheader()
    for row in export_rows:
        writer.writerow(row)

print("Saved:")
print("-", manifest_json)
print("-", manifest_csv)

In [ ]:

# ----------------------------
# Optional contact sheet preview
# ----------------------------

if SAVE_PREVIEW_CONTACT_SHEET and export_rows:
    thumbs = []
    thumb_w, thumb_h = 300, 220
    for row in export_rows:
        img = Image.open(row["preview_path"]).convert("RGB")
        img.thumbnail((thumb_w, thumb_h))
        canvas = Image.new("RGB", (thumb_w, thumb_h + 40), "white")
        x = (thumb_w - img.width) // 2
        y = (thumb_h - img.height) // 2
        canvas.paste(img, (x, y))
        thumbs.append((canvas, row["name"]))

    cols = 3
    rows_n = (len(thumbs) + cols - 1) // cols
    sheet = Image.new("RGB", (cols * thumb_w, rows_n * (thumb_h + 40)), "lightgray")

    from PIL import ImageDraw
    draw = ImageDraw.Draw(sheet)

    for idx, (thumb, name) in enumerate(thumbs):
        r = idx // cols
        c = idx % cols
        x0 = c * thumb_w
        y0 = r * (thumb_h + 40)
        sheet.paste(thumb, (x0, y0))
        draw.text((x0 + 8, y0 + thumb_h + 10), name, fill="black")

    contact_path = reports_dir / "contact_sheet.jpg"
    sheet.save(contact_path, quality=92)
    print("Saved:", contact_path)
else:
    print("Skipped contact sheet")

In [ ]:

# ----------------------------
# Generate Photoshop JSX importer
# ----------------------------

def to_js_string(path_like: str) -> str:
    return path_like.replace("\\", "/").replace('"', '\\"')

if GENERATE_JSX_IMPORTER:
    jsx_path = photoshop_dir / "import_layers_to_psd.jsx"
    base_image_abs = IMAGE_PATH.resolve()
    out_psd_abs = (photoshop_dir / f"{PROJECT_NAME}.psd").resolve()

    layer_entries = []
    for row in export_rows:
        layer_entries.append({
            "name": row["name"],
            "group": row["group"],
            "rgba_path": str(Path(row["rgba_path"]).resolve()),
        })

    layer_entries_js = json.dumps(layer_entries, indent=2)

    jsx = f'''
#target photoshop
app.displayDialogs = DialogModes.NO;

var baseImagePath = "{to_js_string(str(base_image_abs))}";
var outPsdPath = "{to_js_string(str(out_psd_abs))}";
var layerEntries = {layer_entries_js};

function ensureGroup(doc, groupName) {{
    for (var i = 0; i < doc.layerSets.length; i++) {{
        if (doc.layerSets[i].name === groupName) {{
            return doc.layerSets[i];
        }}
    }}
    var g = doc.layerSets.add();
    g.name = groupName;
    return g;
}}

function placeOpenedDocAsLayer(srcDoc, targetDoc, layerName, groupName) {{
    app.activeDocument = srcDoc;
    srcDoc.activeLayer.isBackgroundLayer = false;
    srcDoc.activeLayer.name = layerName;
    srcDoc.activeLayer.duplicate(targetDoc, ElementPlacement.PLACEATBEGINNING);
    srcDoc.close(SaveOptions.DONOTSAVECHANGES);

    app.activeDocument = targetDoc;
    var dupLayer = targetDoc.activeLayer;
    dupLayer.name = layerName;
    var group = ensureGroup(targetDoc, groupName);
    dupLayer.move(group, ElementPlacement.INSIDE);
}}

var baseDoc = app.open(new File(baseImagePath));
baseDoc.flatten();
baseDoc.activeLayer.name = "base_image";

for (var i = 0; i < layerEntries.length; i++) {{
    var entry = layerEntries[i];
    var f = new File(entry.rgba_path);
    if (!f.exists) {{
        continue;
    }}
    var src = app.open(f);
    placeOpenedDocAsLayer(src, baseDoc, entry.name, entry.group);
}}

var psdOpts = new PhotoshopSaveOptions();
psdOpts.layers = true;
psdOpts.alphaChannels = true;
psdOpts.embedColorProfile = true;
baseDoc.saveAs(new File(outPsdPath), psdOpts, true, Extension.LOWERCASE);

alert("PSD created at: " + outPsdPath);
'''
    jsx_path.write_text(jsx, encoding="utf-8")
    print("Saved JSX importer:", jsx_path)
else:
    print("Skipped JSX generation")

In [ ]:

# ----------------------------
# Generate Photoshop prompt sheet
# ----------------------------

prompt_txt = reports_dir / "photoshop_prompts.txt"
with open(prompt_txt, "w", encoding="utf-8") as f:
    for row in export_rows:
        f.write(f"[{row['name']}]\n")
        f.write(f"Label: {row['label']}\n")
        f.write(f"Depth Group: {row['group']}\n")
        f.write(f"Prompt: {row['prompt']}\n")
        f.write(f"Mask: {row['mask_path']}\n")
        f.write(f"Layer: {row['rgba_path']}\n")
        f.write("\n")

print("Saved:", prompt_txt)

In [ ]:

# ----------------------------
# Optional ZIP package
# ----------------------------

if SAVE_ZIP_PACKAGE:
    zip_base = OUT_ROOT.parent / f"{PROJECT_NAME}_package"
    zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=OUT_ROOT)
    print("Created ZIP:", zip_path)
else:
    print("ZIP packaging disabled")

## How to use the exported package in Photoshop

### Manual workflow
1. Open the generated RGBA layer PNGs in Photoshop  
2. Import them into one document  
3. Use the matching full-size masks if you want extra control  
4. Run Generative Fill or ChatGPT-assisted editing on each object  
5. Save the final PSD

### JSX workflow
1. Open Photoshop  
2. Go to **File → Scripts → Browse...**  
3. Select `import_layers_to_psd.jsx`  
4. Photoshop will build a layered PSD automatically

### Suggested ChatGPT/Photoshop prompt style
Use prompts like:
- "Complete the hidden parts of this temple. Keep a clean paper-theater silhouette and transparent background."
- "Extend the deer naturally behind the bushes. Preserve pose and texture."
- "Turn this sky into a full-canvas background layer with smooth transitions."

## Why this solves your layer problem

Each exported asset is:
- the **same size as the original image**
- already isolated onto a **transparent full-canvas layer**
- paired with a **full-canvas mask**
- packaged with a consistent **layer name, prompt, and depth group**

## Optional next upgrade

The next natural step is a **round-trip Photoshop automation**:

1. Notebook exports the full package  
2. Photoshop or UXP plugin edits each layer  
3. Edited layers are saved into a `photoshop_returns/` folder  
4. Notebook re-imports them into the final scene builder

If you want, I can build that next as a second notebook or as an add-on script.